# Notebook 06: v1 vs v2 Prompt Fabrication Comparison Pipeline

End-to-end pipeline comparing **P1 (v1, narrative `.docx`)** vs **P4 (v2, structured JSON `.txt`)**
summaries across 14 confirmed AI fabrication cases.

| Phase | Description |
|-------|-------------|
| **1** | Source document feature extraction — text stats for v1 & v2 |
| **3** | LLM-based fabrication validation — blind detection on both versions |
| **4** | Prompt iteration tracking — feature-level v1 vs v2 improvement table |

Phases 2 (similarity) and 5 (forecasting) are in `scripts/phase2_document_similarity.py`
and `scripts/phase5_timeseries_forecast.py`.

**Key structural difference:**
- v1 → narrative free-text `.docx` (human-readable HPI-style summary)
- v2 → structured JSON `.txt` with `value` + `evidence` per extracted feature

## 0. Setup & Imports

In [ ]:
import os
import sys
import json
import re
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.metrics import precision_score, recall_score, f1_score
from docx import Document
from tqdm.notebook import tqdm

load_dotenv()

PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT",
    r"C:\Users\jamesr4\OneDrive - Memorial Sloan Kettering Cancer Center"
    r"\Documents\GitHub\llm_summarization_br_ca"))
DATA_PRIVATE = Path(os.getenv("DATA_PRIVATE_DIR",
    r"C:\Users\jamesr4\loc\data_private"))
V2_DIR = Path(
    r"C:\Users\jamesr4\OneDrive - Memorial Sloan Kettering Cancer Center"
    r"\Documents\Research\Projects\moo\llm_summary\data\raw\fabrications_iteration_2"
)

FAB_XLSX      = DATA_PRIVATE / "raw" / "ai_fabrications_dataset.xlsx"
V1_PATHS_CSV  = DATA_PRIVATE / "raw" / "v1_summary_paths.csv"
V2_PATHS_CSV  = DATA_PRIVATE / "raw" / "v2_summary_paths.csv"
RUN_OUT_DIR   = PROJECT_ROOT / "experiments" / "runs" / "v1_v2_comparison"
REPORTS_DIR   = PROJECT_ROOT / "reports"

RUN_OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

VALIDATION_MODEL  = "gpt-4o"   # swap to "o1-mini" / "o1" for stronger reasoning
ISSUE_JUDGE_MODEL = "gpt-4o"

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
print(f"OpenAI key loaded : {bool(os.getenv('OPENAI_API_KEY'))}")
print(f"V2 dir exists     : {V2_DIR.exists()}")

: 

## 1. Load Fabrication Cases

In [ ]:
df_fab   = pd.read_excel(FAB_XLSX)
df_v1    = pd.read_csv(V1_PATHS_CSV).rename(columns={"summary_path": "v1_path"})
df_v2    = pd.read_csv(V2_PATHS_CSV).rename(columns={"summary_path": "v2_path"})

ai_cols = [c for c in df_fab.columns if c.endswith("_status_ai")]
for c in ai_cols:
    df_fab[c] = pd.to_numeric(df_fab[c], errors="coerce")

df_fab["surgeon_last"] = df_fab["surgeon"].str.split(",").str[0].str.strip()
df_fab["fab_features"] = df_fab.apply(
    lambda row: [c.replace("_status_ai", "") for c in ai_cols if row[c] == 3], axis=1
)
df_fab["gt_is_fabrication"] = True

df = df_fab.merge(df_v1, on="mrn", how="left").merge(df_v2, on="mrn", how="left")

print(f"Cases: {len(df)}")
print(f"v1 paths found: {df['v1_path'].notna().sum()}")
print(f"v2 paths found: {df['v2_path'].notna().sum()}")

## Phase 1 — Source Document Feature Extraction

Extract and characterize text from both v1 (narrative `.docx`) and v2 (structured JSON `.txt`).

In [ ]:
# ── v1: extract narrative text from .docx ─────────────────────────────────────
def extract_docx_text(path: str) -> str:
    try:
        doc   = Document(str(path))
        parts = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
        for table in doc.tables:
            for trow in table.rows:
                cells = " | ".join(
                    c.text.strip() for c in trow.cells if c.text.strip()
                )
                if cells:
                    parts.append(cells)
        return "\n".join(parts).strip()
    except Exception as exc:
        return f"[ERROR: {exc}]"


# ── v2: parse structured JSON txt ─────────────────────────────────────────────
def parse_v2_json(path: str) -> dict:
    """Return parsed dict from v2 JSON txt. Returns {} on failure."""
    try:
        raw  = Path(path).read_text(encoding="utf-8", errors="ignore")
        return json.loads(raw)
    except json.JSONDecodeError:
        # Some files may have trailing text — try extracting the first JSON block
        try:
            match = re.search(r"\{.*\}", raw, re.DOTALL)
            if match:
                return json.loads(match.group())
        except Exception:
            pass
        return {"_parse_error": True, "_raw": raw[:500]}


def v2_feature_value(parsed: dict, feature_col: str) -> dict:
    """
    Look up a feature in v2 parsed JSON. Searches lesions[], then top-level keys.
    Returns {value, evidence} or {}.
    """
    # Build a flat feature name pattern (e.g. 'invasive_component_size_pathology')
    target = feature_col.replace(" ", "_").lower()

    def _search(obj, depth=0) -> dict | None:
        if depth > 6 or not isinstance(obj, dict):
            return None
        for k, v in obj.items():
            if target in k.lower():
                if isinstance(v, dict) and ("value" in v or "evidence" in v):
                    return v
            sub = _search(v, depth + 1) if isinstance(v, dict) else None
            if sub:
                return sub
            if isinstance(v, list):
                for item in v:
                    sub = _search(item, depth + 1)
                    if sub:
                        return sub
        return None

    result = _search(parsed)
    return result if result else {}


def v2_to_text(parsed: dict) -> str:
    """Flatten v2 JSON to readable text for LLM input."""
    try:
        return json.dumps(parsed, indent=2)[:6000]  # cap for token budget
    except Exception:
        return str(parsed)[:6000]


# ── Extract for all 14 cases ──────────────────────────────────────────────────
df["v1_text"]        = df["v1_path"].apply(
    lambda p: extract_docx_text(p) if pd.notna(p) else ""
)
df["v2_parsed"]      = df["v2_path"].apply(
    lambda p: parse_v2_json(p) if pd.notna(p) and Path(p).exists() else {}
)
df["v2_text"]        = df["v2_parsed"].apply(v2_to_text)

# Extraction status
for _, row in df.iterrows():
    v1_ok = len(row["v1_text"]) > 100
    v2_ok = not row["v2_parsed"].get("_parse_error") and bool(row["v2_parsed"])
    print(f"  MRN {int(row['mrn']):<12} v1={'OK' if v1_ok else 'ERR':<5} "
          f"v2={'OK' if v2_ok else 'ERR':<5} | "
          f"v1_chars={len(row['v1_text']):>5}  v2_keys={len(row['v2_parsed'])}")

In [ ]:
# ── Phase 1: Text statistics ──────────────────────────────────────────────────
def text_stats(text: str) -> dict:
    words = re.findall(r"\b[a-z]+\b", text.lower())
    unique = set(words)
    sents  = [s.strip() for s in re.split(r"[.!?]+", text) if len(s.strip()) > 10]
    return {
        "word_count":       len(words),
        "unique_words":     len(unique),
        "lexical_diversity": round(len(unique) / max(len(words), 1), 3),
        "char_count":       len(text),
        "sentence_count":   len(sents),
    }


stats_rows = []
for _, row in df.iterrows():
    mrn = int(row["mrn"])
    s1  = text_stats(row["v1_text"])
    s2  = text_stats(row["v2_text"])
    stats_rows.append({
        "mrn": mrn,
        "surgeon":          row["surgeon_last"],
        "patient_initials": row["patient_initials"],
        **{f"v1_{k}": v for k, v in s1.items()},
        **{f"v2_{k}": v for k, v in s2.items()},
    })

df_stats = pd.DataFrame(stats_rows)

print("=== Phase 1: Text Statistics ===")
print()
print(df_stats[["mrn", "surgeon", "patient_initials",
               "v1_word_count", "v2_word_count",
               "v1_lexical_diversity", "v2_lexical_diversity"]].to_string(index=False))
print()
print(f"Mean v1 word count: {df_stats['v1_word_count'].mean():.0f}")
print(f"Mean v2 word count: {df_stats['v2_word_count'].mean():.0f}")

In [ ]:
# ── Phase 1 Figure — text stats comparison ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

labels = [f"{row['surgeon'][:6]}/{row['patient_initials']}"
          for _, row in df_stats.iterrows()]
x = np.arange(len(labels))
w = 0.35

# Word count
axes[0].bar(x - w/2, df_stats["v1_word_count"], w, label="v1 (narrative)",
            color="#3498db", edgecolor="white")
axes[0].bar(x + w/2, df_stats["v2_word_count"], w, label="v2 (JSON)",
            color="#e67e22", edgecolor="white")
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=55, ha="right", fontsize=7)
axes[0].set_title("Word Count per Summary", fontweight="bold")
axes[0].set_ylabel("Word Count")
axes[0].legend()

# Lexical diversity
axes[1].bar(x - w/2, df_stats["v1_lexical_diversity"], w, label="v1",
            color="#3498db", edgecolor="white")
axes[1].bar(x + w/2, df_stats["v2_lexical_diversity"], w, label="v2",
            color="#e67e22", edgecolor="white")
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=55, ha="right", fontsize=7)
axes[1].set_title("Lexical Diversity", fontweight="bold")
axes[1].set_ylabel("Unique / Total Words")
axes[1].legend()

# Char count box
axes[2].boxplot(
    [df_stats["v1_char_count"].values, df_stats["v2_char_count"].values],
    labels=["v1 (narrative)", "v2 (JSON)"],
    patch_artist=True,
    boxprops=dict(facecolor="#ecf0f1"),
    medianprops=dict(color="black", linewidth=2),
)
axes[2].set_title("Character Count Distribution", fontweight="bold")
axes[2].set_ylabel("Characters")

plt.suptitle("Phase 1 — Document Text Feature Comparison: v1 vs v2  (n=14 fabrication cases)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "phase1_text_features.png", dpi=150, bbox_inches="tight")
plt.show()

# Save Phase 1 output
df_stats.to_csv(RUN_OUT_DIR / "phase1_text_features.csv", index=False)
print(f"Saved: {RUN_OUT_DIR / 'phase1_text_features.csv'}")

## Phase 3 — LLM Fabrication Validation (v1 vs v2)

**v1 strategy**: send full narrative text + feature name → blind detection  
**v2 strategy**: extract `value` + `evidence` from JSON for the target feature → check accuracy

In [ ]:
FEATURE_DESCRIPTIONS = {
    "lesion_size": (
        "The measured size of the breast lesion as reported in imaging "
        "(mammography, ultrasound, MRI) or pathology."
    ),
    "laterality": "Which breast is affected (left or right).",
    "lesion_location": "Anatomical location of the lesion (quadrant, clock position).",
    "calcifications_asymmetry": "Presence and characterization of calcifications or asymmetry.",
    "additional_enhancement_mri": "Additional enhancement lesions on MRI beyond the index lesion.",
    "extent": "Extent/distribution of disease (focal vs. diffuse, span of calcifications).",
    "accurate_clip_placement": "Whether the biopsy clip is accurately placed at the lesion site.",
    "workup_recommendation": (
        "Additional workup recommended (MRI, staging scans, biopsy, genetic testing) "
        "based on the clinical picture."
    ),
    "Lymph node": "Lymph node status — presence or absence of suspicious axillary lymph nodes.",
    "chronology_preserved": (
        "Whether dates and chronological sequence of events are accurately preserved."
    ),
    "biopsy_method": "Biopsy technique used (US-guided core, stereotactic, surgical excision).",
    "invasive_component_size_pathology": (
        "Size of the invasive carcinoma component as measured on pathology."
    ),
    "histologic_diagnosis": "Pathologic diagnosis / histologic type (IDC, ILC, DCIS, etc.).",
    "receptor": "ER, PR, HER2 receptor status (triple-negative, triple-positive, etc.).",
}

# ── Load actual prompt texts from prompts/frozen/ ─────────────────────────────
PROMPTS_DIR = PROJECT_ROOT / "prompts" / "frozen"

V1_PROMPT_TEXT = (PROMPTS_DIR / "initial_prompt_for_extraction_v1.txt").read_text(
    encoding="utf-8"
)
V2_PROMPT_TEXT = (PROMPTS_DIR / "updated_developer_prompt_feature_extraction_v2.txt").read_text(
    encoding="utf-8"
)

print(f"v1 prompt loaded: {len(V1_PROMPT_TEXT)} chars")
print(f"v2 prompt loaded: {len(V2_PROMPT_TEXT)} chars")
print()
print("v1 technique : template_narrative — NO anti-fabrication constraints")
print("v2 technique : structured_developer_json — explicit anti-fabrication hard constraints")

SYSTEM_PROMPT = """You are a clinical data quality reviewer specializing in breast \
oncology. You review AI-generated clinical summaries for factual accuracy.
Respond ONLY with a JSON object. Do not include any other text."""

print(f"\nFeature descriptions: {len(FEATURE_DESCRIPTIONS)}")

In [ ]:
def _call_llm(messages: list, model: str, use_json_format: bool = True) -> str:
    is_o1 = model.startswith("o1")
    kwargs = {"model": model, "messages": messages}
    if not is_o1 and use_json_format:
        kwargs["response_format"] = {"type": "json_object"}
        kwargs["temperature"] = 0
    resp = client.chat.completions.create(**kwargs)
    return resp.choices[0].message.content.strip()


def _parse_json_response(content: str) -> dict:
    if content.startswith("```"):
        content = "\n".join(content.split("\n")[1:-1])
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        return {"is_fabrication": None, "confidence": None,
                "issue": content, "parse_error": True}


# ── v1: blind detection on narrative text ─────────────────────────────────────
def validate_v1(summary_text: str, feature_name: str, mrn: int) -> dict:
    feat_desc = FEATURE_DESCRIPTIONS.get(feature_name, feature_name.replace("_", " "))
    user_msg  = f"""Review this AI-generated breast oncology summary for fabrication \
in ONE specific clinical feature.

CONTEXT — HOW THIS SUMMARY WAS GENERATED (v1 prompt):
The AI was given a template-style narrative prompt with NO anti-fabrication constraints.
It was instructed to produce a free-text chronological summary organized by modality.
The prompt did NOT require verbatim fidelity, did NOT require "Not reported" for \
missing data, and did NOT have explicit hard constraints against hallucination.
Prompt excerpt (first 500 chars):
---
{V1_PROMPT_TEXT[:500]}
---

FEATURE TO CHECK: {feature_name.replace('_', ' ').upper()}
Feature description: {feat_desc}

AI SUMMARY (v1 — narrative free-text):
---
{summary_text[:4000]}
---

Given that this summary was generated WITHOUT anti-fabrication constraints, assess \
whether the stated value for this feature is clinically implausible, internally \
inconsistent, or appears to be hallucinated. Use your clinical expertise.

Return JSON: {{"is_fabrication": true/false, "confidence": 0-1, \
"issue": "description or empty string"}}"""

    is_o1    = VALIDATION_MODEL.startswith("o1")
    messages = (
        [{"role": "user", "content": SYSTEM_PROMPT + "\n\n" + user_msg}] if is_o1
        else [{"role": "system", "content": SYSTEM_PROMPT},
              {"role": "user",   "content": user_msg}]
    )
    content = _call_llm(messages, VALIDATION_MODEL)
    result  = _parse_json_response(content)
    result.update({"mrn": mrn, "feature_name": feature_name,
                   "version": "v1", "raw_response": content})
    return result


# ── v2: structured JSON validation ────────────────────────────────────────────
def validate_v2(parsed_json: dict, feature_name: str, mrn: int) -> dict:
    feat_desc = FEATURE_DESCRIPTIONS.get(feature_name, feature_name.replace("_", " "))
    feat_data = v2_feature_value(parsed_json, feature_name)

    if not feat_data:
        return {"mrn": mrn, "feature_name": feature_name, "version": "v2",
                "is_fabrication": None, "confidence": None,
                "issue": "Feature not found in v2 JSON", "not_found": True}

    extracted_value    = feat_data.get("value", "NOT FOUND")
    extracted_evidence = feat_data.get("evidence", "NOT FOUND")

    user_msg = f"""Review this AI-extracted clinical feature for fabrication.

CONTEXT — HOW THIS SUMMARY WAS GENERATED (v2 prompt):
The AI was given a formal developer/system prompt with EXPLICIT anti-fabrication \
hard constraints:
  - Use ONLY information explicitly stated in the input text (no inference)
  - Output "Not reported" if missing; "Indeterminate" if conflicting
  - Preserve units and wording VERBATIM (no conversion or normalization)
  - Structured JSON output: each feature has a "value" field and an "evidence" \
field containing the verbatim source text
Prompt excerpt (first 500 chars):
---
{V2_PROMPT_TEXT[:500]}
---

FEATURE: {feature_name.replace('_', ' ').upper()}
Feature description: {feat_desc}

AI EXTRACTED VALUE (v2 — structured JSON):
  value    : {extracted_value}
  evidence : {str(extracted_evidence)[:800]}

Given that this prompt explicitly required verbatim fidelity and prohibited \
inference, assess:
1. Is the extracted value consistent with the verbatim evidence provided?
2. Does the value differ from what the evidence text actually states?
3. Is "Not reported" or "Indeterminate" used appropriately where data is \
missing/conflicting?

Any mismatch between "value" and "evidence" constitutes a fabrication violation \
of the prompt's hard constraints.

Return JSON: {{"is_fabrication": true/false, "confidence": 0-1, \
"issue": "description or empty string", \
"extracted_value": "{extracted_value}"}}"""

    is_o1    = VALIDATION_MODEL.startswith("o1")
    messages = (
        [{"role": "user", "content": SYSTEM_PROMPT + "\n\n" + user_msg}] if is_o1
        else [{"role": "system", "content": SYSTEM_PROMPT},
              {"role": "user",   "content": user_msg}]
    )
    content = _call_llm(messages, VALIDATION_MODEL)
    result  = _parse_json_response(content)
    result.update({"mrn": mrn, "feature_name": feature_name,
                   "version": "v2", "extracted_value": extracted_value,
                   "raw_response": content})
    return result


print("validate_v1() and validate_v2() defined with prompt context")

In [ ]:
# ── Run v1 validation ─────────────────────────────────────────────────────────
V1_CACHE = RUN_OUT_DIR / "phase3_v1_results.json"
V2_CACHE = RUN_OUT_DIR / "phase3_v2_results.json"

v1_results = json.load(open(V1_CACHE)) if V1_CACHE.exists() else {}
v2_results = json.load(open(V2_CACHE)) if V2_CACHE.exists() else {}
print(f"v1 cached: {len(v1_results)}  v2 cached: {len(v2_results)}")

v1_tasks = [
    {"key": f"{int(row['mrn'])}_{feat}",
     "mrn": int(row["mrn"]), "feature": feat,
     "summary_text": row["v1_text"]}
    for _, row in df.iterrows()
    for feat in row["fab_features"]
    if f"{int(row['mrn'])}_{feat}" not in v1_results
]

v2_tasks = [
    {"key": f"{int(row['mrn'])}_{feat}",
     "mrn": int(row["mrn"]), "feature": feat,
     "parsed_json": row["v2_parsed"]}
    for _, row in df.iterrows()
    for feat in row["fab_features"]
    if f"{int(row['mrn'])}_{feat}" not in v2_results
]

print(f"v1 tasks to run: {len(v1_tasks)}  v2 tasks: {len(v2_tasks)}")


def _run_v1(task):
    return task["key"], validate_v1(task["summary_text"], task["feature"], task["mrn"])


def _run_v2(task):
    return task["key"], validate_v2(task["parsed_json"], task["feature"], task["mrn"])


def _run_tasks(tasks, fn, cache_path, cache_dict, desc):
    with ThreadPoolExecutor(max_workers=4) as ex:
        futures = {ex.submit(fn, t): t for t in tasks}
        for fut in tqdm(as_completed(futures), total=len(futures), desc=desc):
            try:
                key, result = fut.result()
                cache_dict[key] = result
                with open(cache_path, "w") as f:
                    json.dump(cache_dict, f, indent=2, default=str)
            except Exception as exc:
                print(f"  ERROR {futures[fut]['key']}: {exc}")


_run_tasks(v1_tasks, _run_v1, V1_CACHE, v1_results, "v1 validation")
_run_tasks(v2_tasks, _run_v2, V2_CACHE, v2_results, "v2 validation")

print(f"\nDone — v1: {len(v1_results)}  v2: {len(v2_results)}")

## Phase 3 — Issue Comparison (Stage 2)

In [ ]:
def validate_issue(model_issue: str, gt_issue: str) -> bool:
    prompt = f"""You are a medical expert reviewing LLM-generated clinical answers.

Compare: does the model-generated issue describe the same clinical inaccuracy as \
the known correct issue? Focus on clinical concept, not exact wording.

Respond with ONLY: True or False

Model issue : {model_issue}
Known issue : {gt_issue}"""

    messages = [{"role": "user", "content": prompt}]
    kwargs   = {"model": ISSUE_JUDGE_MODEL, "messages": messages}
    if not ISSUE_JUDGE_MODEL.startswith("o1"):
        kwargs["temperature"] = 0
    resp = client.chat.completions.create(**kwargs)
    return resp.choices[0].message.content.strip().lower().startswith("true")


ISSUE_CACHE = RUN_OUT_DIR / "phase3_issue_results.json"
issue_results = json.load(open(ISSUE_CACHE)) if ISSUE_CACHE.exists() else {}

issue_tasks = []
for _, row in df.iterrows():
    mrn       = int(row["mrn"])
    gt_issue  = str(row.get("ai_fab_comment", "") or "")
    for feat in row["fab_features"]:
        key = f"{mrn}_{feat}"
        for ver, cache in [("v1", v1_results), ("v2", v2_results)]:
            result = cache.get(key, {})
            ikey   = f"{ver}_{key}"
            if result.get("is_fabrication") is True and ikey not in issue_results:
                issue_tasks.append({
                    "ikey": ikey,
                    "model_issue": str(result.get("issue", "")),
                    "gt_issue":    gt_issue,
                })

print(f"Issue comparison tasks: {len(issue_tasks)}")

def _run_issue(task):
    return task["ikey"], validate_issue(task["model_issue"], task["gt_issue"])

with ThreadPoolExecutor(max_workers=4) as ex:
    futures = {ex.submit(_run_issue, t): t for t in issue_tasks}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Issue judge"):
        try:
            ikey, match = fut.result()
            issue_results[ikey] = match
            with open(ISSUE_CACHE, "w") as f:
                json.dump(issue_results, f, indent=2)
        except Exception as exc:
            print(f"  ERROR: {exc}")

print(f"Done — {len(issue_results)} issue comparisons")

## Phase 4 — Prompt Iteration Tracking & Metrics

In [ ]:
# ── Build combined results table ──────────────────────────────────────────────
result_rows = []
for _, row in df.iterrows():
    mrn       = int(row["mrn"])
    gt_issue  = str(row.get("ai_fab_comment", "") or "")
    for feat in row["fab_features"]:
        key  = f"{mrn}_{feat}"
        r_v1 = v1_results.get(key, {})
        r_v2 = v2_results.get(key, {})
        result_rows.append({
            "mrn":               mrn,
            "surgeon":           row["surgeon_last"],
            "patient_initials":  row["patient_initials"],
            "feature":           feat,
            "gt_is_fabrication": True,
            "gt_issue":          gt_issue,
            # v1
            "v1_pred":           r_v1.get("is_fabrication"),
            "v1_confidence":     r_v1.get("confidence"),
            "v1_issue":          r_v1.get("issue", ""),
            "v1_issue_match":    issue_results.get(f"v1_{key}"),
            # v2
            "v2_pred":           r_v2.get("is_fabrication"),
            "v2_confidence":     r_v2.get("confidence"),
            "v2_issue":          r_v2.get("issue", ""),
            "v2_extracted_val": r_v2.get("extracted_value", ""),
            "v2_issue_match":    issue_results.get(f"v2_{key}"),
        })

df_cmp = pd.DataFrame(result_rows)

# ── Compute metrics per version ────────────────────────────────────────────────
def compute_metrics(pred_series, gt_series):
    mask = pred_series.notna()
    if mask.sum() == 0:
        return {"precision": None, "recall": None, "f1": None,
                "tp": 0, "fp": 0, "fn": 0, "n_eval": 0}
    gt   = gt_series[mask].astype(bool).tolist()
    pred = pred_series[mask].astype(bool).tolist()
    return {
        "precision": round(precision_score(gt, pred, zero_division=0), 3),
        "recall":    round(recall_score(gt, pred, zero_division=0), 3),
        "f1":        round(f1_score(gt, pred, zero_division=0), 3),
        "tp":        sum(g and p for g, p in zip(gt, pred)),
        "fp":        sum(not g and p for g, p in zip(gt, pred)),
        "fn":        sum(g and not p for g, p in zip(gt, pred)),
        "n_eval":    len(gt),
    }


m_v1 = compute_metrics(df_cmp["v1_pred"], df_cmp["gt_is_fabrication"])
m_v2 = compute_metrics(df_cmp["v2_pred"], df_cmp["gt_is_fabrication"])

# Issue match rates
def issue_rate(match_series, pred_series):
    detected = pred_series == True
    n_eval   = match_series[detected].notna().sum()
    n_match  = match_series[detected].sum() if n_eval > 0 else 0
    return round(n_match / n_eval, 3) if n_eval > 0 else None

ir_v1 = issue_rate(df_cmp["v1_issue_match"], df_cmp["v1_pred"])
ir_v2 = issue_rate(df_cmp["v2_issue_match"], df_cmp["v2_pred"])

print("=" * 60)
print(f"  {'Metric':<25} {'v1 (narrative)':<18} {'v2 (JSON)':<18} {'Delta'}")
print("-" * 60)
for metric in ["precision", "recall", "f1"]:
    v1v = m_v1.get(metric)
    v2v = m_v2.get(metric)
    delta = f"{(v2v or 0) - (v1v or 0):+.3f}" if v1v is not None and v2v is not None else "N/A"
    print(f"  {metric:<25} {str(v1v):<18} {str(v2v):<18} {delta}")
print(f"  {'Issue match rate':<25} {str(ir_v1):<18} {str(ir_v2):<18}")
print(f"  {'TP/FN (detected/missed)':<25} {m_v1['tp']}/{m_v1['fn']:<14}   {m_v2['tp']}/{m_v2['fn']}")
print("=" * 60)

In [ ]:
# ── Phase 4 Tracking Table — feature-level improvement ────────────────────────
df_track = df_cmp[[
    "mrn", "surgeon", "patient_initials", "feature",
    "v1_pred", "v2_pred", "gt_is_fabrication", "gt_issue",
    "v1_confidence", "v2_confidence",
    "v1_issue_match", "v2_issue_match",
]].copy()

def status(pred, gt=True):
    if pred is None:
        return "NOT_RUN"
    if pred == gt:
        return "TP"
    return "FN"

df_track["v1_status"] = df_track["v1_pred"].apply(status)
df_track["v2_status"] = df_track["v2_pred"].apply(status)
df_track["improvement"] = (
    (df_track["v1_status"] == "FN") & (df_track["v2_status"] == "TP")
)
df_track["regression"] = (
    (df_track["v1_status"] == "TP") & (df_track["v2_status"] == "FN")
)

print("=== Phase 4 Prompt Iteration Tracking ===")
print(f"Improvements (v1 miss → v2 hit) : {df_track['improvement'].sum()}")
print(f"Regressions  (v1 hit  → v2 miss): {df_track['regression'].sum()}")
print(f"Both correct                     : {((df_track['v1_status']=='TP') & (df_track['v2_status']=='TP')).sum()}")
print(f"Both missed                      : {((df_track['v1_status']=='FN') & (df_track['v2_status']=='FN')).sum()}")
print()
print(df_track[[
    "surgeon", "patient_initials", "feature",
    "v1_status", "v1_confidence", "v2_status", "v2_confidence",
    "improvement", "regression"
]].to_string(index=False))

In [ ]:
# ── Phase 3+4 Figures ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 11))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# A: Detection results side-by-side bar
ax_a = fig.add_subplot(gs[0, 0])
det_data = pd.DataFrame({
    "version": ["v1", "v2"],
    "TP": [m_v1["tp"], m_v2["tp"]],
    "FN": [m_v1["fn"], m_v2["fn"]],
})
x  = np.arange(2)
ax_a.bar(x, det_data["TP"], label="Detected (TP)", color="#e74c3c", edgecolor="white")
ax_a.bar(x, det_data["FN"], bottom=det_data["TP"], label="Missed (FN)",
         color="#95a5a6", edgecolor="white")
ax_a.set_xticks(x)
ax_a.set_xticklabels(["v1 (narrative)", "v2 (JSON)"])
ax_a.set_ylabel("Cases")
ax_a.set_title("Fabrication Detection", fontweight="bold")
ax_a.legend(fontsize=8)

# B: Metrics comparison
ax_b = fig.add_subplot(gs[0, 1])
metrics = ["precision", "recall", "f1"]
v1_vals = [m_v1.get(m, 0) or 0 for m in metrics]
v2_vals = [m_v2.get(m, 0) or 0 for m in metrics]
x2 = np.arange(len(metrics))
ax_b.bar(x2 - 0.2, v1_vals, 0.35, label="v1", color="#3498db", edgecolor="white")
ax_b.bar(x2 + 0.2, v2_vals, 0.35, label="v2", color="#e67e22", edgecolor="white")
for xi, (v1, v2) in zip(x2, zip(v1_vals, v2_vals)):
    ax_b.text(xi - 0.2, v1 + 0.01, f"{v1:.2f}", ha="center", fontsize=7)
    ax_b.text(xi + 0.2, v2 + 0.01, f"{v2:.2f}", ha="center", fontsize=7)
ax_b.set_xticks(x2)
ax_b.set_xticklabels(["Precision", "Recall", "F1"])
ax_b.set_ylim(0, 1.15)
ax_b.set_title("Precision / Recall / F1", fontweight="bold")
ax_b.legend()

# C: Confidence comparison (scatter)
ax_c = fig.add_subplot(gs[0, 2])
conf_cmp = df_cmp.dropna(subset=["v1_confidence", "v2_confidence"])
if len(conf_cmp) > 0:
    colors = conf_cmp["v1_pred"].map(
        {True: "#e74c3c", False: "#2ecc71"}
    ).fillna("#95a5a6")
    ax_c.scatter(conf_cmp["v1_confidence"], conf_cmp["v2_confidence"],
                 c=colors, s=80, edgecolors="white", linewidths=0.5, alpha=0.85)
    ax_c.plot([0, 1], [0, 1], "--", color="gray", alpha=0.5)
    ax_c.set_xlabel("v1 Confidence")
    ax_c.set_ylabel("v2 Confidence")
    ax_c.set_title("Confidence: v1 vs v2", fontweight="bold")
    ax_c.set_xlim(0, 1)
    ax_c.set_ylim(0, 1)

# D: Feature-level improvement heatmap
ax_d = fig.add_subplot(gs[1, :])
hm_data = df_cmp.copy()
hm_data["v1_num"] = hm_data["v1_pred"].map({True: 1, False: 0, None: np.nan})
hm_data["v2_num"] = hm_data["v2_pred"].map({True: 1, False: 0, None: np.nan})
feat_labels = (
    hm_data["feature"]
    .str.replace(r"^feature_\d+_", "", regex=True)
    .str.replace("_", " ")
    .str.title()
)
case_labels = hm_data["surgeon"] + "/" + hm_data["patient_initials"]

combined = pd.DataFrame({
    "case":    case_labels.values,
    "feature": feat_labels.values,
    "v1_det":  hm_data["v1_num"].values,
    "v2_det":  hm_data["v2_num"].values,
})

# Color code: 0=both missed, 1=only v1 detected, 2=only v2 detected, 3=both detected
def combo(v1, v2):
    v1b = bool(v1) if pd.notna(v1) else False
    v2b = bool(v2) if pd.notna(v2) else False
    return v1b * 1 + v2b * 2

combined["combo"] = [combo(r.v1_det, r.v2_det) for r in combined.itertuples()]
cmap = plt.cm.get_cmap("RdYlGn", 4)
ax_d.barh(range(len(combined)),
           np.ones(len(combined)),
           color=[cmap(r.combo / 3) for r in combined.itertuples()],
           edgecolor="white", linewidth=1)
ax_d.set_yticks(range(len(combined)))
ax_d.set_yticklabels(
    [f"{r.case} — {r.feature}" for r in combined.itertuples()], fontsize=7
)
ax_d.set_xlim(0, 1)
ax_d.set_xticks([])

legend_patches = [
    plt.Rectangle((0,0),1,1, color=cmap(0), label="Both missed"),
    plt.Rectangle((0,0),1,1, color=cmap(1/3), label="v1 only detected"),
    plt.Rectangle((0,0),1,1, color=cmap(2/3), label="v2 only detected"),
    plt.Rectangle((0,0),1,1, color=cmap(1),   label="Both detected"),
]
ax_d.legend(handles=legend_patches, loc="lower right", fontsize=8)
ax_d.set_title("Phase 4 — Feature-Level Detection: v1 vs v2 Prompt", fontweight="bold")

fig.suptitle(
    f"Phase 3+4: LLM Fabrication Validation — v1 (narrative) vs v2 (JSON)  "
    f"(model={VALIDATION_MODEL}, n=14 cases)",
    fontsize=12, fontweight="bold"
)
plt.savefig(REPORTS_DIR / "phase3_4_v1_v2_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {REPORTS_DIR / 'phase3_4_v1_v2_comparison.png'}")

In [ ]:
# ── Save all outputs ──────────────────────────────────────────────────────────
from datetime import datetime

df_cmp.to_csv(RUN_OUT_DIR / "phase3_comparison_results.csv", index=False)
df_track.to_csv(RUN_OUT_DIR / "phase4_tracking_table.csv", index=False)

# Iteration summary for Phase 4 tracking (aligns with prompt_metadata.json)
iteration_summary = [
    {
        "iteration":   1,
        "prompt_id":   "P_v1_initial_extraction",
        "version":     "v1",
        "output_format": "narrative_docx",
        "technique":   "template_narrative",
        "anti_fabrication_constraints": False,
        **m_v1,
        "issue_match_rate":             ir_v1,
        "n_improvements_over_baseline": 0,
    },
    {
        "iteration":   2,
        "prompt_id":   "P_v2_developer_extraction",
        "version":     "v2",
        "output_format": "structured_json_txt",
        "technique":   "structured_developer_json",
        "anti_fabrication_constraints": True,
        **m_v2,
        "issue_match_rate":            ir_v2,
        "n_improvements_over_baseline": int(df_track["improvement"].sum()),
        "n_regressions_from_baseline":  int(df_track["regression"].sum()),
    },
]

summary = {
    "run_timestamp":   datetime.utcnow().isoformat(),
    "validation_model": VALIDATION_MODEL,
    "n_cases":          len(df_cmp),
    "prompt_metadata_ref": str(PROJECT_ROOT / "prompts" / "frozen" / "prompt_metadata.json"),
    "iterations":       iteration_summary,
}

with open(RUN_OUT_DIR / "run_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print("=" * 60)
print("OUTPUTS SAVED")
print("=" * 60)
for p in sorted(RUN_OUT_DIR.iterdir()):
    print(f"  {p.name}")
print(f"  reports/phase1_text_features.png")
print(f"  reports/phase3_4_v1_v2_comparison.png")
print()
print("Prompt metadata: prompts/frozen/prompt_metadata.json")
print(f"  v1 → P_v1_initial_extraction  (template_narrative, no constraints)")
print(f"  v2 → P_v2_developer_extraction (structured_developer_json, anti-fab constraints)")